# Stage 2 Notebook 25 - Exp2T Hybrid Q + stage 1 auxiliary geometry supervision

**Why this exists.** Exp2Q (NB22) was the project's best-ever decoded_f1 at 0.039 -- but it didn't actually combine the wins from Exp2N (matched_iou=0.42) and Exp2P (val_lane_f1=0.65). Stage 1's matched_iou stayed at 0.14, the same as pure-query Exp2P. After re-reading the hybrid head's loss path, the cause is clear: **stage 1's coord_pred was exposed in the output dict but not directly supervised by FusionLaneLoss**. The 192 prior curves only got gradient indirectly through stage 2's reading of `per_prior_features`, which optimized those features for stage 2's K=12 ranking task -- not for producing high-quality prior curves.

Exp2T fixes this with a single targeted change: **add an auxiliary FusionLaneLoss term on the hybrid head's `stage1_*` keys**, weighted by `loss.lane.stage1_aux_loss_weight: 0.5`. This forces stage 1 (which is the same CLRKDLaneHead architecture that produced matched_iou=0.42 in Exp2N) to actually train as the geometry champion. Stage 2 (K=12 query refiner) still owns the ranking task end-to-end.

Predicted result: matched_iou should recover to 0.30+ (not necessarily Exp2N's 0.42 since the architecture is still wrapped in a query refiner that pulls the per-prior features toward ranking-friendliness, but substantially better than 0.14). val_lane_f1 should hold near Exp2P/Q's 0.65. **decoded_f1 ~ 0.42 * 0.65 ~ 0.18 if compositions are roughly multiplicative; that's 4-5x our current best.**

Single-knob change vs Exp2Q: `loss.lane.stage1_aux_loss_weight: 0.0 -> 0.5`. All architecture identical.

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After smoke + debug pass, change to `False` for the 10-epoch short run.
3. Output mirrored to notebook cell, Colab runtime log, Drive log file.
4. Do not rerun NB00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp20_rmt_gca_hybrid_with_stage1_aux_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp20_rmt_gca_hybrid_with_stage1_aux_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp20_rmt_gca_hybrid_with_stage1_aux_joint_smoke.log
OK exp20_rmt_gca_hybrid_with_stage1_aux_joint.yaml
  lane_shape=(1, 12, 72, 2) det_shape=(1, 4, 4)
  lane_loss=3.3262 det_loss=3.8919 grad_cos=-0.0337 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.4997878074645996, 'gate/lane_mean': 0.4976291358470917, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp20_rmt_gca_hybrid_with_stage1_aux_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short10'
    EPOCHS = 10
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp20_rmt_gca_hybrid_with_stage1_aux_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp20_rmt_gca_hybrid_with_stage1_aux_joint_short10 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp20_rmt_gca_hybrid_with_stage1_aux_joint_short10.tar --epochs 10 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp20_rmt_gca_hybrid_with_stage1_aux_joint_short10.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp20_rmt_gca_hybrid_with_stage1_aux_joint_short10_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp20_rmt_gca_hybrid_with_stage1_aux_joint.yaml --curve-tar /content/d

0

## What to watch in Exp2T training

Reference Exp2Q (NB22, hybrid no aux):
- val_lane_f1=0.643, matched_iou=0.142, decoded_f1=0.039, oracle_f1=0.074.

The new `epoch_summary` line will include `lane/stage1_aux_total`, `lane/stage1_aux_cls`, `lane/stage1_aux_iou`, `lane/stage1_aux_reg` (visible via the metrics JSON; not all printed in epoch_summary).

Pass criteria at epoch 10:

- **`val/matched_line_iou >= 0.30`**: stage 1 should recover toward Exp2N's 0.42 (probably won't fully reach it under joint training, but 2-3x Exp2Q's 0.14 is the target).
- **`val/lane_exist_best_f1 >= 0.55`**: stage 2 ranking should hold near Exp2P/Q's 0.65.
- **`val/lane/decoded_f1 >= 0.10`**, ideally `>= 0.15`: this is the metric that matters. Exp2Q's 0.039 was the previous best; we want at least 3x.
- **`val/lane/decoded_oracle_f1 >= 0.18`**: oracle ranking on the K=12 outputs should reflect the lifted geometry quality.
- `val/det/metric_map50` shouldn't regress further (should hold ~0.003).

Failure signals -> next ablation:

- Geometry doesn't recover (matched_iou < 0.20): the K=12 query refiner is corrupting the per-prior features even with explicit stage 1 supervision. Try freezing stage 1 entirely after a head_warmup phase.
- Cls regresses (best_f1 < 0.45): stage 1's cls supervision conflicts with stage 2's. Set stage1_aux on cls components to 0 (only supervise stage 1 geometry).
- Both wins fail: hybrid pattern itself is the wrong direction; pivot to Exp2S (Bezier) results.